In [1]:
# imports the environment variables

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
pip install pypdf

Note: you may need to restart the kernel to use updated packages.


In [3]:
# LOADING THE PDF 
#PyPDFLoader - extracts text from pdf , one page at a time 
#DirectoryLoader - scans folder and finds everything matching the filetye

from langchain_community.document_loaders import PyPDFLoader , DirectoryLoader

DATA_PATH = "data/"

def load_pdf_files(data):
    loader = DirectoryLoader(data , glob = "*.pdf" , loader_cls = PyPDFLoader)
    documents = loader.load()
    return documents 

documents = load_pdf_files(DATA_PATH)

print(f"number of pages loaded : {len(documents)}")

C:\Users\APPU\AppData\Local\Temp\ipykernel_29968\3547835385.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader , DirectoryLoader


number of pages loaded : 759


In [4]:
#checking if it works 
print(documents[0])

page_content='The GALE
ENCYCLOPEDIA
of MEDICINE
SECOND EDITION' metadata={'producer': 'GPL Ghostscript 9.10', 'creator': '', 'creationdate': '2017-05-01T10:37:35-07:00', 'moddate': '2017-05-01T10:37:35-07:00', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'source': 'data\\The_GALE_ENCYCLOPEDIA_of_MEDICINE_SECOND.pdf', 'total_pages': 759, 'page': 0, 'page_label': '1'}


In [5]:
# CHUNKING
#ResursiveCharacterTextSplitter - is "recursive" because it tries to cut at the best possible boundary: first tries paragraph 
#breaks (\n\n), then line breaks (\n), then sentences, then words, only falling back to a hard character cut as a last resort

from langchain_text_splitters import RecursiveCharacterTextSplitter

def create_chunks(extracted_data) : 
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 500 , # max ~500 characters per chunk (not words, not tokens — raw characters)
        chunk_overlap = 50 # the last 50 characters of one chunk get repeated as the first 50 characters of the next chunk.
    )
    text_chunks = text_splitter.split_documents(extracted_data)
    return text_chunks

text_chunks = create_chunks(documents)
print("number of chunks created : " , len(text_chunks))

number of chunks created :  7080


In [6]:
pip install langchain_huggingface

Note: you may need to restart the kernel to use updated packages.


In [7]:
# embeddings - turninf text into vectors 
from langchain_huggingface import HuggingFaceEmbeddings

def get_embedding_model():
    embedding_model = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")
    return embedding_model

embedding_model = get_embedding_model()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
#build and save FAISS index 

from langchain_community.vectorstores import FAISS

DB_FAISS_PATH = "vectorstore/db_faiss"
db = FAISS.from_documents(text_chunks , embedding_model)
db.save_local(DB_FAISS_PATH)

print("FAISS index saved to : " , DB_FAISS_PATH)

FAISS index saved to :  vectorstore/db_faiss
